# 03 - EDA: Titanic

Análise exploratória do dataset do Titanic.

Objetivo: praticar limpeza de dados, tratamento de valores nulos e análise de sobrevivência.

Dataset: 891 passageiros com informações como idade, sexo, classe da cabine e se sobreviveram ou não.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

## Nível 1 — Exploração e limpeza

### Quantas linhas e colunas tem o dataset? Quais colunas têm valores nulos e quantos?
#### → .shape, .isnull().sum()

In [7]:
print('Linhas e Colunas:', df.shape)


Linhas e Colunas: (891, 12)


In [11]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

### A coluna Age tem muitos nulos. Preencha os valores nulos com a mediana da idade. Por que mediana e não média?
#### → .fillna(), .median() — pesquise a diferença entre média e mediana

## 📊 Média vs Mediana 

A **média** é sensível a outliers, o que pode acabar puxando o valor muito para cima ou para baixo, algo que não queremos aqui.

Ja a **mediana** ela representa melhor a fatia central das pessoas que estavam no titanic, sendo uma representação melhor nessa situação.

---

In [26]:
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### A coluna Cabin tem muitos nulos também. Nesse caso, em vez de preencher, delete a coluna inteira. Quando faz sentido deletar vs preencher?
#### → .drop(columns=[ ])

## Por que deletamos a coluna Cabin?

A coluna `Cabin` possui muitos valores nulos, mais da metade da coluna são valores nulos, o que é um corte muito pequeno para se tirar uma análise certeira, 

então é melhor retirar essa coluna da analise, ja que tambem **nao é uma coluna tao relevante** para a analise atual.

A regra geral é:
- **Menos de 50% de nulos** → vale considerar preencher (imputação)
- **Mais de 50% de nulos** → o volume de dados ausentes compromete a análise,
  sendo mais seguro remover a coluna

---

In [36]:
df = df.drop(columns='Cabin')

### A coluna Embarked tem apenas 2 nulos. Preencha com o valor mais frequente (moda).
#### → .fillna(), .mode() — .mode() retorna uma Series, pesquise como pegar só o primeiro valor

In [46]:
embarked_mode = df['Embarked'].mode()
df['Embarked'] = df['Embarked'].fillna(embarked_mode[0])
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S
...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,W./C. 6607,23.4500,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C


### Confirme que não existe mais nenhum valor nulo no dataset após as limpezas.
#### → .isnull().sum()

In [47]:
df.isnull().sum() # verifica se sobrou algum nulo

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

## Nível 2 — Análise de sobrevivência

### Qual a taxa geral de sobrevivência? (coluna Survived: 1 = sobreviveu)
#### → .mean()

Apenas **38%** dos passageiros do Titanic sobreviveram, enquanto **62%** dos passageiros vieram a óbito!

In [55]:
print('Passageiros que sobreviveram:', round(df['Survived'].mean(), 2))
print('Passageiros que morreram:', round(1 - df['Survived'].mean(), 2))

Passageiros que sobreviveram: 0.38
Passageiros que morreram: 0.62


### Qual a taxa de sobrevivência por sexo? Mulheres sobreviveram mais que homens?
#### → .groupby()[ ].mean()

Apenas **19%** dos homens sobreviveram ao acidente, o que é um valor muito baixo perto dos **74%** de sobreviventes mulheres

In [76]:
round(df.groupby('Sex')[['Survived']].mean(), 2)

,Survived
Sex,
female,0.74
male,0.19


### Qual a taxa de sobrevivência por classe (Pclass 1, 2, 3)? A classe social influenciou as chances?
#### → .groupby()[ ].mean().sort_values()

O que podemos observar é que a classe social dos passageiros foi algo que afetou a **taxa de sobrevivência** dos passageiros.
onde podemos ver uma queda progressiva clara!

- Classe 1 (mais rica) → 62% sobreviveram
- Classe 2 → (classe média) → 47% sobreviveram
- Classe 3 (mais pobre) → 24% sobreviveram

In [81]:
round(df.groupby('Pclass')[['Survived']].mean().sort_values(by='Pclass'), 2)

,Survived
Pclass,
1,0.63
2,0.47
3,0.24


### Compare a idade média dos sobreviventes vs não sobreviventes. Havia diferença de idade?
#### → .groupby()[ ].mean()

In [89]:
round(df.groupby('Survived')[['Age']].mean().rename(index={0: 'Não sobreviveu', 1: 'Sobreviveu'}))

,Age
Survived,
Não sobreviveu,30.0
Sobreviveu,28.0


### Filtre apenas os passageiros da primeira classe que não sobreviveram. Quantos são? Qual a idade média deles?
#### → df[ (condição1) & (condição2) ]

## Nível 3 — Visualização

### Gráfico de barras: taxa de sobrevivência por sexo.
#### → .groupby()[ ].mean().plot(kind= )

### Gráfico de barras: taxa de sobrevivência por classe (1, 2, 3).
#### → .groupby()[ ].mean().plot(kind= )

### Histograma da distribuição de idades. Aplique as 4 perguntas de leitura de gráfico e escreva sua observação em Markdown.
#### → .plot(kind='hist')

## Desafio bônus


### Crie um gráfico de barras mostrando a taxa de sobrevivência por sexo E classe ao mesmo tempo. Dica: você vai precisar agrupar por duas colunas simultaneamente.
#### → .groupby([ , ])[ ].mean().unstack().plot(kind= )